In [1]:
from pathlib import Path
import pandas as pd

DATA_FOLDER = Path("/home/user/Documents/zepto")

files = sorted(DATA_FOLDER.glob("*.csv"))

print(f"Found {len(files)} files")

for f in files:
    print(f.name)

Found 12 files
zepto_2026_07_04.csv
zepto_2026_07_07.csv
zepto_2026_07_08.csv
zepto_2026_07_09.csv
zepto_2026_07_10.csv
zepto_2026_07_14.csv
zepto_2026_07_15.csv
zepto_2026_07_16.csv
zepto_2026_07_20.csv
zepto_2026_07_21.csv
zepto_2026_07_22.csv
zepto_2026_07_23.csv


In [2]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("/home/user/Documents/zepto")

dfs = {}

for file in sorted(DATA_FOLDER.glob("*.csv")):
    print(f"Loading {file.name}...")

    df = pd.read_csv(file, low_memory=False)

    # filename
    df["source_file"] = file.name

    # crawl date
    df["crawl_date"] = pd.to_datetime(
        df["crawl_date_and_time"]
    ).dt.date

    dfs[file.stem] = df

print(f"\nLoaded {len(dfs)} datasets.")

Loading zepto_2026_07_04.csv...
Loading zepto_2026_07_07.csv...
Loading zepto_2026_07_08.csv...
Loading zepto_2026_07_09.csv...
Loading zepto_2026_07_10.csv...
Loading zepto_2026_07_14.csv...
Loading zepto_2026_07_15.csv...
Loading zepto_2026_07_16.csv...
Loading zepto_2026_07_20.csv...
Loading zepto_2026_07_21.csv...
Loading zepto_2026_07_22.csv...
Loading zepto_2026_07_23.csv...

Loaded 12 datasets.


In [3]:
for name, df in dfs.items():
    print(name, df.shape)

zepto_2026_07_04 (1037455, 33)
zepto_2026_07_07 (1043504, 33)
zepto_2026_07_08 (1026016, 33)
zepto_2026_07_09 (1049475, 33)
zepto_2026_07_10 (711405, 33)
zepto_2026_07_14 (881077, 33)
zepto_2026_07_15 (553752, 33)
zepto_2026_07_16 (530232, 33)
zepto_2026_07_20 (552107, 33)
zepto_2026_07_21 (539534, 33)
zepto_2026_07_22 (443539, 33)
zepto_2026_07_23 (165326, 33)


In [4]:
summary = []

for name, df in dfs.items():
    summary.append({
        "Date": name.replace("zepto_", ""),
        "Rows": len(df),
        "Unique IDs": df["unique_id"].nunique(),
        "Unique URLs": df["product_url"].nunique(),
        "Stores": df["store_id"].nunique(),
        "Store Locations": df["store_name_location"].nunique(),
        "Cities": df["city"].nunique(),
        "Departments": df["department"].nunique(),
        "Categories": df["category"].nunique(),
        "Sub Categories": df["sub_category"].nunique(),
        "Brands": df["brand_name"].nunique(),
        "OOS %": round((df["out_of_stock_flag"]=="Yes").mean()*100,2)
    })

summary = pd.DataFrame(summary)
summary

,Date,Rows,Unique IDs,Unique URLs,Stores,Store Locations,Cities,Departments,Categories,Sub Categories,Brands,OOS %
0,2026_07_04,1037455,17349,17351,381,292,54,10,28,161,1766,18.66
1,2026_07_07,1043504,16351,16351,375,295,48,10,27,160,1687,21.01
2,2026_07_08,1026016,17374,17374,357,287,57,10,27,159,1714,21.25
3,2026_07_09,1049475,16566,16566,369,291,56,10,27,159,1697,20.14
4,2026_07_10,711405,8855,8855,253,259,6,10,26,158,1226,21.09
5,2026_07_14,881077,8824,8825,310,292,6,10,24,159,1222,18.49
6,2026_07_15,553752,14936,14940,206,147,34,10,23,159,1622,21.03
7,2026_07_16,530232,14804,14804,198,146,34,10,23,159,1624,21.61
8,2026_07_20,552107,15827,15827,196,146,42,10,24,159,1626,22.60
9,2026_07_21,539534,16636,16636,194,147,46,10,24,160,1680,22.60


In [5]:
missing_summary = {}

for name, df in dfs.items():
    missing_summary[name] = df.isna().sum()

missing_df = pd.DataFrame(missing_summary)
missing_df

,zepto_2026_07_04,zepto_2026_07_07,zepto_2026_07_08,zepto_2026_07_09,zepto_2026_07_10,zepto_2026_07_14,zepto_2026_07_15,zepto_2026_07_16,zepto_2026_07_20,zepto_2026_07_21,zepto_2026_07_22,zepto_2026_07_23
crawl_date_and_time,0,0,0,0,0,0,0,0,0,0,0,0
platform_name,0,0,0,0,0,0,0,0,0,0,0,0
store_id,0,0,0,0,0,0,0,0,0,0,0,0
store_name_location,0,0,0,0,0,0,0,0,0,0,0,0
city,0,0,0,0,0,0,0,0,0,0,0,0
pin_code,0,0,0,0,0,0,0,0,0,0,0,0
unique_id,0,0,0,0,0,0,0,0,0,0,0,0
product_id_upc_ean,0,0,0,0,0,0,0,0,0,0,0,0
department,0,0,0,0,0,0,0,0,0,0,0,0
category,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
# Get dataset names in chronological order
keys = sorted(dfs.keys())

for i in range(len(keys)-1):

    d1 = set(dfs[keys[i]]["department"].dropna())
    d2 = set(dfs[keys[i+1]]["department"].dropna())

    print(f"\n{keys[i]} -> {keys[i+1]}")

    print("Added:", sorted(d2-d1))
    print("Removed:", sorted(d1-d2))



zepto_2026_07_04 -> zepto_2026_07_07
Added: []
Removed: []

zepto_2026_07_07 -> zepto_2026_07_08
Added: []
Removed: []

zepto_2026_07_08 -> zepto_2026_07_09
Added: []
Removed: []

zepto_2026_07_09 -> zepto_2026_07_10
Added: []
Removed: []

zepto_2026_07_10 -> zepto_2026_07_14
Added: []
Removed: []

zepto_2026_07_14 -> zepto_2026_07_15
Added: []
Removed: []

zepto_2026_07_15 -> zepto_2026_07_16
Added: []
Removed: []

zepto_2026_07_16 -> zepto_2026_07_20
Added: []
Removed: []

zepto_2026_07_20 -> zepto_2026_07_21
Added: []
Removed: []

zepto_2026_07_21 -> zepto_2026_07_22
Added: []
Removed: []

zepto_2026_07_22 -> zepto_2026_07_23
Added: []
Removed: []


In [9]:
for i in range(len(keys)-1):

    c1 = set(dfs[keys[i]]["category"].dropna())
    c2 = set(dfs[keys[i+1]]["category"].dropna())

    print(f"\n{keys[i]} -> {keys[i+1]}")

    print("Added:", sorted(c2-c1))
    print("Removed:", sorted(c1-c2))


zepto_2026_07_04 -> zepto_2026_07_07
Added: []
Removed: ['Paan Corner']

zepto_2026_07_07 -> zepto_2026_07_08
Added: []
Removed: []

zepto_2026_07_08 -> zepto_2026_07_09
Added: []
Removed: []

zepto_2026_07_09 -> zepto_2026_07_10
Added: ['Jewellery']
Removed: ['Protein & Nutrition', 'Unlisted1']

zepto_2026_07_10 -> zepto_2026_07_14
Added: []
Removed: ['Home Needs', 'Stationery & Books']

zepto_2026_07_14 -> zepto_2026_07_15
Added: []
Removed: ['Jewellery']

zepto_2026_07_15 -> zepto_2026_07_16
Added: []
Removed: []

zepto_2026_07_16 -> zepto_2026_07_20
Added: ['Stationery & Books']
Removed: []

zepto_2026_07_20 -> zepto_2026_07_21
Added: []
Removed: []

zepto_2026_07_21 -> zepto_2026_07_22
Added: ['Jewellery']
Removed: []

zepto_2026_07_22 -> zepto_2026_07_23
Added: []
Removed: ['Jewellery']


In [10]:
for i in range(len(keys)-1):

    s1 = set(dfs[keys[i]]["sub_category"].dropna())
    s2 = set(dfs[keys[i+1]]["sub_category"].dropna())

    print(f"\n{keys[i]} -> {keys[i+1]}")

    print("Added:", sorted(s2-s1))
    print("Removed:", sorted(s1-s2))


zepto_2026_07_04 -> zepto_2026_07_07
Added: ['Tampons & Menstrual Cups']
Removed: ['Sets', 'Toothpaste & Mouthwash']

zepto_2026_07_07 -> zepto_2026_07_08
Added: []
Removed: ['Tampons & Menstrual Cups']

zepto_2026_07_08 -> zepto_2026_07_09
Added: []
Removed: []

zepto_2026_07_09 -> zepto_2026_07_10
Added: ['Sets']
Removed: ['Desserts', 'Hydration Drinks']

zepto_2026_07_10 -> zepto_2026_07_14
Added: ['Crafts & Hobby', 'Tampons & Menstrual Cups']
Removed: ['Handwash & Sanitizers']

zepto_2026_07_14 -> zepto_2026_07_15
Added: ['Desserts', 'Hydration Drinks']
Removed: ['Sets', 'Tampons & Menstrual Cups']

zepto_2026_07_15 -> zepto_2026_07_16
Added: []
Removed: []

zepto_2026_07_16 -> zepto_2026_07_20
Added: []
Removed: []

zepto_2026_07_20 -> zepto_2026_07_21
Added: ['Handwash & Sanitizers', 'Non-Alcoholic & Energy Drink']
Removed: ['Hydration Drinks']

zepto_2026_07_21 -> zepto_2026_07_22
Added: ['Sets']
Removed: ['Desserts', 'Handwash & Sanitizers', 'Non-Alcoholic & Energy Drink']

ze

In [11]:
store_summary = []

for name, df in dfs.items():

    store_summary.append(
        df.groupby("store_id")
          .size()
          .reset_index(name="Products")
          .assign(Date=name)
    )

store_summary = pd.concat(store_summary)

In [14]:
store_summary

,store_id,Products,Date
0,0053a04d-aeb4-46b5-99fe-d740491e3843,3192,zepto_2026_07_04
1,0059ff6a-7eb0-477a-a7f5-69256f2c444b,1471,zepto_2026_07_04
2,005dcc9a-d50c-442f-ae5e-f89f35d1a01a,3131,zepto_2026_07_04
3,01185b5e-b794-4cfb-99c9-625397e548ac,3287,zepto_2026_07_04
4,02d2c306-7699-434b-947e-88c99afc54f9,3200,zepto_2026_07_04
5,02dc6b57-1dfe-4efe-a57b-4324a830de78,3740,zepto_2026_07_04
6,06031ad9-0300-4c0d-bc7d-cc16959c7467,3997,zepto_2026_07_04
7,06bc0cca-b09b-4c18-b6c5-8358f91c7201,3735,zepto_2026_07_04
8,07a6d7c0-aea9-434c-846e-9308e495db6a,1362,zepto_2026_07_04
9,085097a9-564c-45fe-98d8-23d7fd268328,3689,zepto_2026_07_04


In [13]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [15]:
for name, df in dfs.items():

    print("\n",name)

    print(
        df["city"]
        .value_counts()
    )


 zepto_2026_07_04
city
Bangalore                    143769
Hyderabad                    130638
Mumbai                        97070
Chennai                       64877
Noida                         58013
Gurgaon                       57178
Delhi                         56647
Kolkata                       48950
Jaipur                        34092
Pune                          32363
Ghaziabad                     28080
Lucknow                       24681
Ahmedabad                     22038
Coimbatore                    18122
Kochi                         11872
Nashik                        11502
Chandigarh                    10977
Ludhiana                      10884
Faridabad                     10686
Vijayawada                    10551
Madurai                        7605
Dehradun                       7455
Kanpur                         7188
Nagpur                         7163
Prayagraj                      7157
Indore                         7120
Surat                          7116
Pani

In [16]:
for name, df in dfs.items():

    print("\n",name)

    print(
        df["department"]
        .value_counts()
    )


 zepto_2026_07_04
department
Beauty & Cosmetics      170644
Personal Care           156382
Baby Care               141434
Dairy & Breakfast       114853
Organic & Premium       112800
Cold Drinks & Juices     92494
Bakery & Biscuits        79453
Sweet Tooth              77042
Munchies                 50866
Atta, Rice & Dal         41487
Name: count, dtype: int64

 zepto_2026_07_07
department
Beauty & Cosmetics      170012
Personal Care           156673
Baby Care               136158
Dairy & Breakfast       116735
Organic & Premium       116567
Cold Drinks & Juices     93711
Bakery & Biscuits        80102
Sweet Tooth              79767
Munchies                 51940
Atta, Rice & Dal         41839
Name: count, dtype: int64

 zepto_2026_07_08
department
Beauty & Cosmetics      168568
Personal Care           153398
Baby Care               136134
Organic & Premium       113805
Dairy & Breakfast       113343
Cold Drinks & Juices     91875
Bakery & Biscuits        78933
Sweet Tooth          

In [17]:
for name, df in dfs.items():

    print("\n",name)

    print(
        df["category"]
        .value_counts()
    )


 zepto_2026_07_04
category
Skincare                     98629
Dairy, Bread & Eggs          70444
Baby Care                    68881
Toys & Sports                64053
Atta, Rice, Oil & Dals       63267
Bath & Body                  63010
Cold Drinks & Juices         62576
Ice Creams & More            61396
Munchies                     58750
Biscuits                     54813
Hair Care                    44387
Sweet Cravings               43199
Packaged Food                42500
Makeup & Beauty              41954
Breakfast & Sauces           41780
Feminine Hygiene             38551
Masala, Dry Fruits & More    33320
Tea, Coffee & More           33064
Fragrances & Grooming        26556
Pharma & Wellness             9294
Frozen Food                   6494
Zepto Cafe                    4979
Electronics & Appliances      4677
Home Needs                     490
Stationery & Books             238
Unlisted1                       84
Protein & Nutrition             65
Paan Corner                

In [18]:
for name, df in dfs.items():

    print("\n",name)

    print(
        df["sub_category"]
        .value_counts()
    )


 zepto_2026_07_04
sub_category
Face Wash & Scrubs            25144
Face Creams & Gels            14707
Chips & Crisps                14530
Toothbrush & More             14319
Rice & More                   13593
Body Lotion & Moisturizer     13463
Chocolates                    11968
Perfumes                      10918
Tubs                          10709
Sticks                        10413
Baby Bath                      9921
Premium Chocolates             9501
Dry Fruits & Nuts Munchies     9356
Namkeens                       9221
Gifting                        9106
Cones                          8994
Cups                           8779
Sun Care                       8757
Cookies                        8715
Energy Bars                    8699
Baking Mixes & Ingredients     8632
Cold Coffee & Iced Tea         8622
Milk Drinks                    8606
Besan, Sooji & Maida           8566
Dry Fruits & Nuts              8550
Honey & Spreads                8532
Breakfast Cereals              8

In [19]:
summary = []

for name, df in dfs.items():
    summary.append({
        "Date": name.replace("zepto_", ""),
        "Rows": len(df),
        "Cities": df["city"].nunique(),
        "Stores": df["store_id"].nunique(),
        "Store Locations": df["store_name_location"].nunique(),
        "Departments": df["department"].nunique(),
        "Categories": df["category"].nunique(),
        "Sub Categories": df["sub_category"].nunique(),
        "Brands": df["brand_name"].nunique(),
        "Unique Products (unique_id)": df["unique_id"].nunique(),
        "Unique URLs": df["product_url"].nunique()
    })

summary_df = pd.DataFrame(summary)

summary_df

,Date,Rows,Cities,Stores,Store Locations,Departments,Categories,Sub Categories,Brands,Unique Products (unique_id),Unique URLs
0,2026_07_04,1037455,54,381,292,10,28,161,1766,17349,17351
1,2026_07_07,1043504,48,375,295,10,27,160,1687,16351,16351
2,2026_07_08,1026016,57,357,287,10,27,159,1714,17374,17374
3,2026_07_09,1049475,56,369,291,10,27,159,1697,16566,16566
4,2026_07_10,711405,6,253,259,10,26,158,1226,8855,8855
5,2026_07_14,881077,6,310,292,10,24,159,1222,8824,8825
6,2026_07_15,553752,34,206,147,10,23,159,1622,14936,14940
7,2026_07_16,530232,34,198,146,10,23,159,1624,14804,14804
8,2026_07_20,552107,42,196,146,10,24,159,1626,15827,15827
9,2026_07_21,539534,46,194,147,10,24,160,1680,16636,16636
